# 1-3절 연습 문제 풀이

이 노트북은 1-3절 연습 문제(1-8 ~ 1-12)의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `code_examples/ch01/01-03_example.ipynb`를 참고한다.
- 위에서부터 차례대로 실행한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
# 환경 설정 - 시드 고정 (예제 노트북과 같은 SEED를 사용해 같은 데이터를 만든다)
import random

import numpy as np
import torch

SEED = 8
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 공통 준비

본문 1-3절의 데이터, 모델, 손실 함수, 학습 루프를 그대로 옮겨 온다.
다만 학습 로그를 에포크마다 출력하면 문제 하나에 수백 줄이 나오므로, 마지막 결과만 돌려주는 조용한 학습 함수를 함께 둔다.

In [2]:
SAMPLE_SIZE = 50    # 생성할 샘플의 수
TRUE_A = 4.9        # 중력 가속도(9.8)의 절반
TRUE_B = 0.0        # 초기 낙하거리
LR = 0.0001         # 학습률
EPOCHS = 15         # 전체 학습 에포크

def generate_true_data(true_a=TRUE_A, true_b=TRUE_B, sample_size=SAMPLE_SIZE):
    X = torch.rand((sample_size, 1)) * 10          # 관측시간
    Y_ideal = true_a * X ** 2 + true_b             # 이상적인 낙하거리
    noise = torch.randn((sample_size, 1)) * Y_ideal * 0.1   # 관측 오차
    return X, Y_ideal + noise

def model(parameters, X):
    return parameters[0] * X ** 2 + parameters[1]

def criterion(Y_pred, Y_true):
    return torch.mean((Y_pred - Y_true) ** 2)

def train_quiet(parameters, X, Y_true, epochs=EPOCHS, learning_rate=LR,
                model=model, criterion=criterion):
    """[코드 1-32]와 같은 학습 루프. 로그 대신 마지막 손실만 반환한다."""
    for _ in range(epochs):
        parameters.grad = None
        train_loss = criterion(model(parameters, X), Y_true)
        train_loss.backward()
        with torch.no_grad():
            parameters -= learning_rate * parameters.grad
    return train_loss.item()

X, Y_true = generate_true_data()
TRAIN_SIZE = int(SAMPLE_SIZE * 0.8)
X_train, Y_train = X[:TRAIN_SIZE], Y_true[:TRAIN_SIZE]
X_test, Y_test = X[TRAIN_SIZE:], Y_true[TRAIN_SIZE:]
print(f'훈련 데이터 {X_train.shape[0]}개, 평가 데이터 {X_test.shape[0]}개')

훈련 데이터 40개, 평가 데이터 10개


## 연습 문제 1-8

> 파라미터의 값을 무작위로 초기화하는 대신, a의 초깃값으로 0, 5, 10을, b의 초깃값으로 -3, 0, 3을 사용하는
> 아홉 개의 서로 다른 파라미터 초깃값으로 모델을 학습한 후 결과를 확인해 보자.
> 그 결과를 바탕으로 파라미터 초깃값과 모델 학습 결과를 설명해 보자.

In [3]:
A_INITS = [0., 5., 10.]
B_INITS = [-3., 0., 3.]

print(f'{"a 초깃값":>8} {"b 초깃값":>8} | {"학습 후 a":>10} {"학습 후 b":>10} {"훈련 손실":>12}')
print('-' * 58)
for a_init in A_INITS:
    for b_init in B_INITS:
        parameters = torch.tensor([a_init, b_init], requires_grad=True)
        train_loss = train_quiet(parameters, X_train, Y_train)
        print(f'{a_init:8.1f} {b_init:8.1f} | {parameters[0].item():10.3f} {parameters[1].item():10.3f} {train_loss:12.2f}')

   a 초깃값    b 초깃값 |     학습 후 a     학습 후 b        훈련 손실
----------------------------------------------------------
     0.0     -3.0 |      5.158     -2.906       385.22
     0.0      0.0 |      5.106      0.089       380.34
     0.0      3.0 |      5.055      3.083       384.46
     5.0     -3.0 |      5.171     -2.992       384.95
     5.0      0.0 |      5.119      0.002       379.83
     5.0      3.0 |      5.068      2.997       383.70
    10.0     -3.0 |      5.184     -3.078       385.70
    10.0      0.0 |      5.132     -0.084       380.33
    10.0      3.0 |      5.081      2.911       383.95


### 풀이 해설

결과에서 두 가지가 드러난다.

**a는 초깃값과 무관하게 5.1 부근으로 모인다.** 0에서 시작하든 10에서 시작하든 15에포크 뒤에는 비슷한 값에 도달한다.
a의 기울기가 크기 때문이다. 학습 초기 a의 기울기는 만 단위인데, 이 정도면 학습률 0.0001을 곱해도 한 번에 1 이상 움직인다.

**b는 초깃값에서 거의 움직이지 않는다.** -3에서 시작하면 -2.9 언저리에, 3에서 시작하면 3.08 언저리에 머문다.
b의 기울기는 a의 기울기보다 50배 넘게 작아서, 같은 학습률로는 15에포크 동안 의미 있게 움직이지 못한다.
b의 이상적인 값이 0인데도 b = -3에서 시작한 모델은 여전히 -2.9에 있다.

즉 **초깃값의 영향은 파라미터마다 다르다.** 기울기가 큰 파라미터는 초깃값을 금방 잊지만,
기울기가 작은 파라미터는 초깃값에 오래 붙들려 있는다. 본문이 "무작위로 정해진 파라미터 초깃값의 영향도 있다"고
말한 것의 실제 내용이 이것이다. 손실이 가장 낮은 조합이 b를 0에서 시작한 경우라는 점도 같은 이야기다.

### 문제 검토

- **적절성: 적합. 1장에서 가장 잘 설계된 문제 중 하나다.** 초깃값 아홉 조합이라는 구체적 지시 덕분에
  독자가 헤매지 않고, 결과 표를 보면 위 해설의 두 가지 현상이 저절로 눈에 들어온다.
  본문 각주 12가 이 문제를 가리키고 있어 연결도 자연스럽다.
- **[검토] 두 번째 문장의 요구가 막연하다.** "파라미터 초깃값과 모델 학습 결과를 설명해 보자"만으로는
  무엇을 봐야 하는지 알기 어렵다. 이 문제의 진짜 관찰 대상은 **a와 b가 서로 다르게 움직인다**는 점인데,
  a만 보고 "초깃값은 상관없다"로 결론 내고 넘어갈 수 있다.

**윤문안**

> **1-8**. 파라미터의 값을 무작위로 초기화하는 대신, a의 초깃값으로 0, 5, 10을, b의 초깃값으로 -3, 0, 3을 사용하는
> 아홉 개의 서로 다른 파라미터 초깃값으로 모델을 학습한 후 결과를 확인해 보자.
> 그리고 학습을 마친 a와 b가 초깃값의 영향을 각각 얼마나 받았는지 비교하고, 그 차이가 생긴 이유를 설명해 보자.

## 연습 문제 1-9

> 1-3절의 예제에서 샘플의 수를 충분히 늘려, 학습으로 구한 두 파라미터의 값이 4.9와 0에 가까워지는지 확인해 보자.
>
> 힌트: 샘플의 수에 따라 전체 학습 에포크도 수정해야 한다.

In [4]:
SAMPLE_SETTINGS = [(50, 15), (500, 200), (5000, 500)]   # (샘플 수, 전체 학습 에포크)

print(f'{"샘플 수":>8} {"에포크":>8} | {"학습 후 a":>10} {"학습 후 b":>10}')
print('-' * 46)
for sample_size, epochs in SAMPLE_SETTINGS:
    torch.manual_seed(SEED)
    X_all, Y_all = generate_true_data(sample_size=sample_size)
    train_size = int(sample_size * 0.8)
    parameters = torch.tensor([1., 0.], requires_grad=True)
    train_quiet(parameters, X_all[:train_size], Y_all[:train_size], epochs=epochs)
    print(f'{sample_size:8d} {epochs:8d} | {parameters[0].item():10.4f} {parameters[1].item():10.4f}')

print(f'\n이상적인 값: a = {TRUE_A}, b = {TRUE_B}')

    샘플 수      에포크 |     학습 후 a     학습 후 b
----------------------------------------------
      50       15 |     5.1089     0.0714
     500      200 |     4.8025     0.1101
    5000      500 |     4.9194     0.0521

이상적인 값: a = 4.9, b = 0.0


### 풀이 해설

샘플 수를 50에서 5,000으로 늘리면 a가 4.92까지 내려와 이상적인 값 4.9에 가까워진다.
샘플이 50개일 때는 5.1 부근에 머물렀는데, 이는 적은 표본에 우연히 섞인 노이즈의 치우침을 모델이 그대로 학습했기 때문이다.
표본이 많아질수록 노이즈가 서로 상쇄되어 데이터가 참값 곡선을 더 정확히 드러낸다.

힌트가 말하는 대로 에포크도 함께 늘려야 한다. 샘플만 늘리고 15에포크에 머물면 학습이 끝나기 전에 멈춘다.
다만 b는 샘플을 늘려도 0에 아주 가까워지지는 않는다. 앞 문제에서 본 대로 b의 기울기가 작아
같은 학습률로는 천천히 움직이기 때문이다.

한편 손실값 자체는 샘플을 늘려도 작아지지 않는다. 이 데이터의 노이즈는 낙하거리에 비례하도록 만들어져 있어,
관측시간이 큰 샘플이 많아질수록 오차의 절대 크기도 커지기 때문이다. **손실의 크기와 파라미터의 정확도는 별개**다.

### 문제 검토

- **적절성: 적합.** 본문이 "데이터와 에포크 수를 충분히 늘리면 4.9와 0.0으로 수렴한다"고 단언한 대목을
  독자가 직접 확인하게 하는 문제다. 확인 가능한 주장만 책에 남긴다는 점에서 좋은 배치다.
- **[검토] b는 0에 잘 가까워지지 않는다.** 실제로 해 보면 a는 4.92까지 내려오지만 b는 0.05 안팎에 머문다.
  문제가 "두 파라미터의 값이 4.9와 0에 가까워지는지"를 묻고 있어, 독자는 b에서 기대와 다른 결과를 보고
  자기 구현을 의심하게 된다. 이것이 오히려 [연습 문제 1-8]에서 배운 '파라미터마다 기울기 크기가 다르다'와
  이어지는 좋은 관찰거리이므로, 지문에서 한 번 짚어 주는 편이 낫다.

**윤문안**

> **1-9**. 1-3절의 예제에서 샘플의 수를 충분히 늘려, 학습으로 구한 두 파라미터의 값이 4.9와 0에 가까워지는지
> 확인해 보자. 두 파라미터가 가까워지는 정도에 차이가 있다면 그 이유도 생각해 보자.

## 연습 문제 1-10

> 모델 학습에 사용하는 손실 함수를 평균절대오차 손실 함수로 바꿔 모델을 학습하고, 손실 함수별 결과를 비교해 보자.

In [5]:
def criterion_mae(Y_pred, Y_true):
    """평균절대오차 손실 함수"""
    return torch.mean(torch.abs(Y_pred - Y_true))

# 본문과 같은 학습률(0.0001)로 평균절대오차를 사용해 학습
parameters = torch.tensor([1., 0.], requires_grad=True)
train_quiet(parameters, X_train, Y_train, criterion=criterion_mae)
print(f'평균절대오차, 학습률 {LR}  -> a = {parameters[0].item():.4f}, b = {parameters[1].item():.4f}')

# 비교: 같은 조건에서 평균제곱오차를 사용한 경우
parameters = torch.tensor([1., 0.], requires_grad=True)
train_quiet(parameters, X_train, Y_train)
print(f'평균제곱오차, 학습률 {LR}  -> a = {parameters[0].item():.4f}, b = {parameters[1].item():.4f}')

평균절대오차, 학습률 0.0001  -> a = 1.0432, b = 0.0015
평균제곱오차, 학습률 0.0001  -> a = 5.1089, b = 0.0714


In [6]:
# 두 손실 함수의 기울기 크기를 비교해 본다
for name, loss_fn in (('평균제곱오차', criterion), ('평균절대오차', criterion_mae)):
    parameters = torch.tensor([1., 0.], requires_grad=True)
    loss = loss_fn(model(parameters, X_train), Y_train)
    loss.backward()
    print(f'{name}: 손실 {loss.item():12.2f}, a의 기울기 {parameters.grad[0].item():12.2f}')

평균제곱오차: 손실     28602.18, a의 기울기    -13701.29
평균절대오차: 손실       118.74, a의 기울기       -28.80


In [7]:
# 평균절대오차에 맞게 학습률을 키워 다시 학습
LR_MAE = 0.01
parameters = torch.tensor([1., 0.], requires_grad=True)
train_quiet(parameters, X_train, Y_train, learning_rate=LR_MAE, criterion=criterion_mae)
print(f'평균절대오차, 학습률 {LR_MAE} -> a = {parameters[0].item():.4f}, b = {parameters[1].item():.4f}')

평균절대오차, 학습률 0.01 -> a = 4.9570, b = 0.1115


### 풀이 해설

**본문의 학습률을 그대로 쓰면 평균절대오차 모델은 거의 학습되지 않는다.** a가 1.0에서 1.04로 움직이는 데 그친다.
같은 조건의 평균제곱오차 모델이 5.11까지 가는 것과 대조적이다.

이유는 두 손실 함수의 기울기 크기 차이다. 위 비교에서 보듯 같은 지점에서 평균제곱오차의 기울기는 만 단위인데
평균절대오차의 기울기는 수십 단위다. 평균절대오차는 오차를 제곱하지 않으므로 오차가 아무리 커도
기울기에 그 크기가 반영되지 않기 때문이다.

**학습률을 100배로 키우면(0.01) 평균절대오차 모델도 15에포크 만에 a = 4.96에 도달한다.**
오히려 평균제곱오차보다 이상적인 값 4.9에 가깝다. 노이즈가 큰 샘플에 덜 끌려간 결과로,
본문이 말한 "평균절대오차는 데이터에 노이즈가 많아도 덜 흔들린다"가 그대로 나타난 것이다.

여기서 얻을 교훈은 **손실 함수를 바꾸면 학습률도 함께 조정해야 한다**는 점이다.
학습률은 손실 함수와 독립적인 값이 아니다.

### 문제 검토

- **[중요] 본문 설정 그대로는 문제가 풀리지 않는다.** 이 문제의 지시는 "손실 함수를 바꿔 학습하고 비교"인데,
  학습률 0.0001을 그대로 두면 평균절대오차 모델은 사실상 학습되지 않는다(a = 1.04).
  독자는 이 결과를 보고 "평균절대오차는 쓸모없는 손실 함수"라고 잘못 결론 내리기 쉽다.
  본문 p29가 "평균절대오차는 안정적인 모델을 만드는 데 유리하다"고 설명한 것과 정반대의 인상을 준다.
- **[검토] 그런데 이 함정이 곧 이 문제의 가치다.** 학습률을 키워 보는 과정에서
  '손실 함수가 바뀌면 기울기의 크기가 달라지고, 따라서 학습률도 달라져야 한다'는 것을 스스로 발견하게 된다.
  이는 5장 이후 옵티마이저를 배울 때까지 계속 쓰이는 감각이다.
  따라서 **문제를 바꾸기보다 지문에 학습률을 함께 조정해 보라는 안내를 넣는 것**을 권한다.

**윤문안**

> **1-10**. 모델 학습에 사용하는 손실 함수를 평균절대오차 손실 함수로 바꿔 모델을 학습하고,
> 손실 함수별 결과를 비교해 보자. 학습이 잘 되지 않는다면 학습률을 조정해 보고, 두 손실 함수에 알맞은 학습률이
> 왜 서로 다른지도 생각해 보자.

## 연습 문제 1-11

> 1-3절의 예제는 딥러닝으로 데이터를 y = ax² + b에 맞춰 a와 b의 값을 찾는 회귀 분석 모델이다.
> 그런데 이 데이터가 몇 차의 함수로 표현되는 모델에 적합한지 미리 알 수 없다면 지수까지 파라미터로 두는 모델을
> 사용할 수 있다. 회귀 분석 모델을 y = ax^c + b로 바꾸고 a, b, c 세 파라미터의 값을 찾도록 예제를 수정해 보자.

In [8]:
def model_with_exponent(parameters, X):
    """y = a * x^c + b 형태의 모델. parameters는 [a, b, c] 순서다."""
    return parameters[0] * X ** parameters[2] + parameters[1]

# 먼저 본문과 같은 방식(파라미터마다 같은 학습률)으로 학습해 본다
parameters = torch.tensor([1., 0., 2.], requires_grad=True)
train_quiet(parameters, X_train, Y_train, model=model_with_exponent)
print(f'모든 파라미터에 학습률 {LR} -> a = {parameters[0].item()}, b = {parameters[1].item()}, c = {parameters[2].item()}')

모든 파라미터에 학습률 0.0001 -> a = nan, b = nan, c = nan


In [9]:
# 세 파라미터의 기울기 크기를 비교해 본다
parameters = torch.tensor([1., 0., 2.], requires_grad=True)
loss = criterion(model_with_exponent(parameters, X_train), Y_train)
loss.backward()
for name, grad in zip(['a', 'b', 'c'], parameters.grad.tolist()):
    print(f'{name}의 기울기: {grad:15.2f}')

a의 기울기:       -13701.29
b의 기울기:         -237.49
c의 기울기:       -28595.64


In [10]:
# 파라미터마다 다른 학습률을 적용해 학습
LR_PER_PARAM = torch.tensor([0.0001, 0.0001, 0.0000001])   # a, b는 본문과 같게, c만 아주 작게
EPOCHS_LONG = 2000

parameters = torch.tensor([1., 0., 2.], requires_grad=True)
for _ in range(EPOCHS_LONG):
    parameters.grad = None
    train_loss = criterion(model_with_exponent(parameters, X_train), Y_train)
    train_loss.backward()
    with torch.no_grad():
        parameters -= LR_PER_PARAM * parameters.grad

print(f'파라미터별 학습률, {EPOCHS_LONG}에포크')
print(f'    a = {parameters[0].item():.4f}, b = {parameters[1].item():.4f}, c = {parameters[2].item():.4f}')
print(f'    훈련 손실 = {train_loss.item():.2f}')

파라미터별 학습률, 2000에포크
    a = 5.2312, b = 0.1191, c = 1.9895
    훈련 손실 = 378.34


### 풀이 해설

모델 함수를 바꾸는 것 자체는 한 줄이면 된다. 어려운 부분은 **학습이 되게 만드는 것**이다.

본문과 같은 학습률 0.0001을 세 파라미터에 똑같이 적용하면 몇 에포크 만에 값이 `nan`이 된다.
기울기 비교에서 보듯 c의 기울기가 a, b보다 훨씬 크기 때문이다.
c는 지수 자리에 있어서 조금만 움직여도 예측값이 몇 배로 달라진다.
여기에 a, b와 같은 학습률을 곱하면 한 걸음이 너무 커져 손실이 발산한다.

그래서 **파라미터마다 다른 학습률**을 적용했다. c에만 0.0000001을 쓰고 에포크를 2,000으로 늘리면
c가 1.989로, a가 5.23으로 수렴한다. 데이터가 실제로 2차 함수에서 만들어졌으므로 c가 2에 가까워지는 것이 맞다.
훈련 손실도 378 부근으로, 지수를 2로 고정한 본문 모델(380)과 비슷한 수준까지 내려온다.

학습률을 파라미터마다 다르게 주는 것은 임시방편이다. 5장에서 만나는 Adam 같은 옵티마이저가
이 일을 자동으로 해 준다. 이 문제는 그런 도구가 왜 필요한지를 미리 겪게 해 준다.

### 문제 검토

- **[중요] 지시한 대로 고치면 학습이 발산한다.** "회귀 분석 모델을 y = ax^c + b로 바꾸고 예제를 수정해 보자"를
  글자 그대로 따르면(모델 함수만 바꾸고 나머지는 그대로) 손실이 `nan`이 되어 아무 결과도 얻지 못한다.
  학습률을 파라미터별로 나누거나 아주 작게 낮추고 에포크를 수백 배로 늘려야 비로소 결과가 나온다.
  1장을 갓 뗀 독자가 스스로 이 지점에 도달하기는 어렵다.
- **[검토] 난도 대비 표시가 없다.** 1장의 다른 도전 문제(1-5, 1-7, 1-12)에는 [도전 문제] 표시가 있는데
  이 문제에는 없다. 실제 난도는 1-12 못지않다. **[도전 문제] 표시를 붙이고 힌트를 다는 것**을 권한다.
- **교육적 가치는 높다.** 파라미터마다 적절한 학습률이 다를 수 있다는 것은 옵티마이저의 존재 이유이고,
  그것을 1장 도구만으로 몸으로 겪게 한다. 문제를 빼기보다 힌트를 붙여 살리는 편이 낫다.

**윤문안**

> **1-11**. [도전 문제] 1-3절의 예제는 딥러닝으로 데이터를 y = ax² + b에 맞춰 a와 b의 값을 찾는 회귀 분석 모델이다.
> 그런데 이 데이터가 몇 차의 함수로 표현되는 모델에 적합한지 미리 알 수 없다면 지수까지 파라미터로 두는 모델을
> 사용할 수 있다. 회귀 분석 모델을 y = ax^c + b로 바꾸고 a, b, c 세 파라미터의 값을 찾도록 예제를 수정해 보자.
>
> 힌트: 세 파라미터의 기울기를 출력해 크기를 비교해 보자. 모든 파라미터에 같은 학습률을 쓰면 학습이 발산한다.

## 연습 문제 1-12 [도전 문제]

> 학습 곡선을 보면 학습 초반에는 손실이 크고, 학습이 진행되면서 점차 줄어든다.
> '손실이 클 때는 강하게, 손실이 작을 때는 약하게 학습한다.'는 발상은 딥러닝에만 국한되지 않는 일반적인 원리다.
> 1-3절의 예제에 이 아이디어를 적용해 보자.

In [11]:
BASE_LR = 0.0002    # 기준 학습률: 초반에 크게 움직이도록 본문 학습률의 2배로 잡는다

def train_scheduled(parameters, X, Y_true, epochs=EPOCHS, base_lr=BASE_LR):
    """손실이 줄어드는 만큼 학습률도 함께 줄이는 학습 루프"""
    first_loss = None
    for _ in range(epochs):
        parameters.grad = None
        train_loss = criterion(model(parameters, X), Y_true)
        train_loss.backward()
        if first_loss is None:
            first_loss = train_loss.item()
        # 손실이 첫 손실에서 줄어든 비율의 제곱근만큼 학습률을 줄인다
        learning_rate = base_lr * (train_loss.item() / first_loss) ** 0.5
        with torch.no_grad():
            parameters -= learning_rate * parameters.grad
    return train_loss.item()

for name, train_fn in (('고정 학습률', train_quiet), ('손실에 따라 조정', train_scheduled)):
    parameters = torch.tensor([1., 0.], requires_grad=True)
    train_loss = train_fn(parameters, X_train, Y_train)
    with torch.no_grad():
        test_loss = criterion(model(parameters, X_test), Y_test).item()
    print(f'{name:12} -> a = {parameters[0].item():.4f}, 훈련 손실 = {train_loss:8.2f}, 평가 손실 = {test_loss:8.2f}')

고정 학습률       -> a = 5.1089, 훈련 손실 =   380.16, 평가 손실 =   803.60
손실에 따라 조정    -> a = 4.9017, 훈련 손실 =   473.29, 평가 손실 =   586.05


In [12]:
# 학습률을 손실에 그대로 비례시키면 어떻게 되는지 확인
parameters = torch.tensor([1., 0.], requires_grad=True)
first_loss = None
for epoch in range(EPOCHS):
    parameters.grad = None
    train_loss = criterion(model(parameters, X_train), Y_train)
    train_loss.backward()
    if first_loss is None:
        first_loss = train_loss.item()
    with torch.no_grad():
        parameters -= 0.001 * (train_loss.item() / first_loss) * parameters.grad
print(f'학습률을 크게 잡고 손실에 비례시킨 경우 -> a = {parameters[0].item()}')

학습률을 크게 잡고 손실에 비례시킨 경우 -> a = nan


### 풀이 해설

손실이 줄어드는 비율의 제곱근만큼 학습률을 줄이도록 학습 루프를 고쳤다.
초반에 더 강하게 학습하도록 기준 학습률은 본문의 2배(0.0002)로 잡았다.

결과를 보면 **평가 손실이 803에서 586으로 크게 낮아진다.** 학습으로 구한 a도 4.90으로
이상적인 값 4.9에 거의 정확히 도달한다. 초반에는 크게 움직여 빠르게 접근하고, 손실이 줄어든 뒤에는
보폭을 줄여 훈련 데이터의 노이즈에 덜 끌려간 결과다.
훈련 손실은 오히려 고정 학습률보다 높은데(473 대 380), 이는 훈련 데이터에 덜 맞춰졌다는 뜻이므로
[그림 1-11]에서 본 과적합 관점에서는 좋은 신호다.

다만 이 아이디어에는 함정이 있다. **경사하강법은 이미 이 원리를 어느 정도 내장하고 있다.**
평균제곱오차의 기울기는 오차에 비례하므로, 손실이 크면 기울기도 크고 따라서 한 걸음도 크다.
[연습 문제 1-10]에서 확인했듯 손실이 28,602일 때 a의 기울기는 -13,701이지만,
손실이 400 부근으로 내려오면 기울기도 -398로 함께 작아진다. 학습률은 그대로인데 걸음 폭이 저절로 줄어든다.

그래서 학습률까지 손실에 비례시키면 같은 효과가 두 번 곱해진다.
마지막 코드처럼 학습률을 손실에 그대로(제곱근 없이) 비례시키고 기준 학습률을 조금만 키우면 손실이 발산해 `nan`이 된다.
제곱근을 씌운 것은 이 이중 반영을 완화하기 위해서다.

학습이 진행됨에 따라 학습률을 조절하는 기법은 **학습률 스케줄링**이라는 이름으로 실제로 널리 쓰인다.
다만 이 실험처럼 기준 학습률과 줄이는 속도를 함께 맞춰야 효과를 본다.

### 문제 검토

- **적절성: 도전 문제로 적합하되, 성공 기준이 모호하다.** "이 아이디어를 적용해 보자"만으로는
  무엇을 만들면 푼 것인지 알 수 없다. 구현 방법도 여러 갈래이고, 적용 전후를 무엇으로 비교해야 하는지도 없다.
- **[검토] 정직한 결과가 '큰 차이 없음'이다.** 위 실험에서 훈련 손실은 거의 같고 평가 손실만 나아진다.
  독자가 기대한 극적인 개선이 나오지 않으므로, 구현이 틀렸다고 생각하고 그만두기 쉽다.
  그런데 **왜 극적으로 좋아지지 않는지**(경사하강법이 이미 같은 일을 하고 있다)가 이 문제에서 얻을 수 있는
  가장 값진 통찰이다. 지문이 그 방향을 살짝 가리켜 주면 문제의 가치가 크게 올라간다.

**윤문안**

> **1-12**. [도전 문제] 학습 곡선을 보면 학습 초반에는 손실이 크고, 학습이 진행되면서 점차 줄어든다.
> '손실이 클 때는 강하게, 손실이 작을 때는 약하게 학습한다.'는 발상은 딥러닝에만 국한되지 않는 일반적인 원리다.
> 1-3절의 예제에 이 아이디어를 적용하고, 적용하기 전과 후의 훈련 손실과 평가 손실을 비교해 보자.
> 기대만큼 크게 달라지지 않는다면, 경사하강법이 파라미터를 갱신하는 식을 다시 보며 그 이유를 생각해 보자.